# VeriTone Tier 2: Kathbath-first preparation and training

Run this notebook top-to-bottom in Colab or Kaggle. It uses the Hugging Face streaming route directly:

```python
from datasets import load_dataset

ds = load_dataset('ai4bharat/Kathbath', 'telugu', split='train', streaming=True)
```

Kathbath is genuine Indian speech. It does not contain spoof labels, so this notebook never relabels it as spoof. It prepares genuine data and starts Tier 2 training only when a real spoof dataset is also available. Tier 1 is not modified or executed.

In [1]:
# 1. Runtime and dependency setup
import os, platform, shutil, subprocess, sys

print('Python:', sys.version)
print('Platform:', platform.platform())
print('Disk free GB:', round(shutil.disk_usage('/').free / 1e9, 2))
print(subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout or 'nvidia-smi unavailable')

!pip -q install 'datasets>=2.20' 'huggingface_hub>=0.24' 'transformers>=4.40' soundfile
import torch
print('Torch:', torch.__version__)
print('CUDA:', torch.cuda.is_available(), 'count:', torch.cuda.device_count())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM GB:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2))

Python: 3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0]
Platform: Linux-6.6.122+-x86_64-with-glibc2.39
Disk free GB: 69.63
Sat Sep 12 03:16:42 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   53C    P8             14W /   70W |       0MiB /  15360MiB |      0%      Default |
|             

In [2]:
# 2. Get the repository and configure authenticated Hugging Face access.
from pathlib import Path
import sys, os, getpass

# The google.colab package can exist in local environments; use Colab's runtime marker.
IN_COLAB = bool(os.environ.get('COLAB_RELEASE_TAG'))
if IN_COLAB:
    REPO = '/content/VeriTone'
    %cd /content
    if not Path(REPO).exists():
        !git clone https://github.com/TSK-3/VeriTone.git VeriTone
else:
    candidates = [Path.cwd(), Path.cwd().parent, Path(r'c:\Users\theka\Downloads\sih')]
    repo_path = next((path for path in candidates if (path / 'pyproject.toml').exists()), None)
    if repo_path is None:
        raise FileNotFoundError('Local VeriTone repository not found; open the notebook from the repository workspace')
    REPO = str(repo_path)
%cd $REPO
# Do not install the [ml] extra here: it can replace Colab's CUDA-enabled PyTorch.
!pip -q install -e '.[dev,tier2]'
sys.path.insert(0, REPO + '/src')

# Use an environment variable, an existing Hugging Face cache, or a hidden prompt.
from huggingface_hub import get_token, login
token = os.environ.get('HF_TOKEN') or get_token()
if IN_COLAB:
    try:
        from google.colab import userdata
        token = token or userdata.get('HF_TOKEN')
    except Exception:
        pass
if not token:
    print('No Hugging Face token found in this notebook kernel.')
    print('Create a read token at https://huggingface.co/settings/tokens with Kathbath access.')
    token = getpass.getpass('Paste your Hugging Face token (input is hidden): ').strip()
if not token:
    raise RuntimeError('No Hugging Face token supplied; Kathbath access requires authentication.')
os.environ['HF_TOKEN'] = token
login(token=token, add_to_git_credential=False)
print('Authenticated Hugging Face session configured')
print('Repository ready:', REPO, 'Colab:', IN_COLAB)

/content
/content/VeriTone
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for voice-clone-detection (pyproject.toml) ... done


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Authenticated Hugging Face session configured
Repository ready: /content/VeriTone Colab: True


## 3. Load Kathbath through Hugging Face streaming

The next cell intentionally uses the same Telugu route that was already verified. It then inspects the five requested language configurations without downloading the full dataset.

In [3]:
# 4. Direct streaming load and schema inspection.
from datasets import load_dataset

LANGUAGE_CONFIGS = {
    'telugu': 'te',
    'hindi': 'hi',
    'tamil': 'ta',
    'kannada': 'kn',
    'malayalam': 'ml',
}

# This is the exact verified starting route.
ds = load_dataset(
    'ai4bharat/Kathbath',
    'telugu',
    split='train',
    streaming=True
)
print(ds)
print('features:', ds.features)
first = next(iter(ds))
print('first record keys:', first.keys())
print('first record:', first)

# Inspect only one record per requested language; this does not download the full dataset.
streaming_sets = {'telugu': ds}
for config_name in LANGUAGE_CONFIGS:
    if config_name == 'telugu':
        continue
    probe = load_dataset('ai4bharat/Kathbath', config_name, split='train', streaming=True)
    streaming_sets[config_name] = probe
    print(config_name, 'features:', probe.features)
    print(config_name, 'sample keys:', next(iter(probe)).keys())

Resolving data files:   0%|          | 0/22 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/33 [00:00<?, ?it/s]

IterableDataset({
    features: ['fname', 'text', 'audio_filepath', 'lang', 'duration', 'gender', 'speaker_id'],
    num_shards: 33
})
features: {'fname': Value('string'), 'text': Value('string'), 'audio_filepath': Audio(sampling_rate=None, decode=True, num_channels=None, stream_index=None), 'lang': Value('string'), 'duration': Value('float64'), 'gender': Value('string'), 'speaker_id': Value('int64')}
first record keys: dict_keys(['fname', 'text', 'audio_filepath', 'lang', 'duration', 'gender', 'speaker_id'])
first record: {'fname': '844424932551735-484-f.m4a', 'text': 'గతంలో స్థానిక ప్రజా ప్రతినిధులు మెట్రో రైలును పొడిగించేందుకు హామీలు ఇవ్వడంతో కోటి ఆశలు పెంచుకున్నారు', 'audio_filepath': <datasets.features._torchcodec.AudioDecoder object at 0x7926427f8440>, 'lang': 'te', 'duration': 6.3390625, 'gender': 'Female', 'speaker_id': 484}


Resolving data files:   0%|          | 0/22 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/32 [00:00<?, ?it/s]

...

Resolving data files:   0%|          | 0/33 [00:00<?, ?it/s]

kannada features: {'fname': Value('string'), 'text': Value('string'), 'audio_filepath': Audio(sampling_rate=None, decode=True, num_channels=None, stream_index=None), 'lang': Value('string'), 'duration': Value('float64'), 'gender': Value('string'), 'speaker_id': Value('int64')}
kannada sample keys: dict_keys(['fname', 'text', 'audio_filepath', 'lang', 'duration', 'gender', 'speaker_id'])


Resolving data files:   0%|          | 0/22 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/33 [00:00<?, ?it/s]

malayalam features: {'fname': Value('string'), 'text': Value('string'), 'audio_filepath': Audio(sampling_rate=None, decode=True, num_channels=None, stream_index=None), 'lang': Value('string'), 'duration': Value('float64'), 'gender': Value('string'), 'speaker_id': Value('int64')}
malayalam sample keys: dict_keys(['fname', 'text', 'audio_filepath', 'lang', 'duration', 'gender', 'speaker_id'])


## 5. Prepare genuine and generated-spoof pilots

Kathbath records remain genuine. The next cell materializes genuine WAVs. A following cell generates a clearly labeled spoof pilot from Kathbath transcripts using the real multilingual MMS TTS models. Generated files are not presented as human recordings; their generator is recorded in metadata.

In [4]:
# 6. Materialize a small genuine-only pilot and write provenance.
import csv, shutil
from pathlib import Path
import numpy as np
import soundfile as sf
import torch
import torch.nn.functional as F

PILOT_ROWS_PER_LANGUAGE = 20
TARGET_SAMPLE_RATE = 16000
DATA_ROOT = Path(REPO) / 'data' / 'kathbath_pilot'
GENUINE_ROOT = DATA_ROOT / 'genuine'
GENUINE_ROOT.mkdir(parents=True, exist_ok=True)
provenance_path = DATA_ROOT / 'provenance.csv'
metadata_path = DATA_ROOT / 'metadata.csv'


def write_kathbath_audio(row, target):
    """Write a 16 kHz mono WAV from a path-backed row or torchcodec AudioDecoder."""
    audio_value = row.get('audio_filepath') or row.get('audio')
    if audio_value is None:
        raise ValueError('Kathbath row has neither audio_filepath nor audio')

    if isinstance(audio_value, (str, Path)):
        source_path = Path(str(audio_value))
        if not source_path.is_file():
            raise FileNotFoundError(f'Kathbath audio path is not local: {source_path}')
        data, sample_rate = sf.read(str(source_path), dtype='float32', always_2d=False)
    else:
        if not hasattr(audio_value, 'get_all_samples'):
            raise TypeError(f'Unsupported Kathbath audio value: {type(audio_value)!r}')
        samples = audio_value.get_all_samples()
        data = samples.data
        sample_rate = int(samples.sample_rate)
        if hasattr(data, 'detach'):
            data = data.detach().cpu().numpy()
        data = np.asarray(data)

    if data.ndim == 2:
        data = data.mean(axis=0 if data.shape[0] <= 8 else 1)
    data = np.asarray(data).reshape(-1).astype(np.float32)
    if sample_rate != TARGET_SAMPLE_RATE:
        waveform = torch.from_numpy(data).view(1, 1, -1)
        target_length = max(1, round(data.size * TARGET_SAMPLE_RATE / sample_rate))
        data = F.interpolate(waveform, size=target_length, mode='linear', align_corners=False).view(-1).numpy()
    sf.write(str(target), data, TARGET_SAMPLE_RATE, subtype='PCM_16')
    return TARGET_SAMPLE_RATE, data.size / TARGET_SAMPLE_RATE


prepared = []
for config_name, language_code in LANGUAGE_CONFIGS.items():
    dataset = streaming_sets[config_name]
    for index, row in enumerate(dataset):
        if index >= PILOT_ROWS_PER_LANGUAGE:
            break
        source_name = str(row.get('fname') or f'{language_code}_{index:05d}.wav')
        target = GENUINE_ROOT / language_code / Path(source_name).name
        target = target.with_suffix('.wav')
        target.parent.mkdir(parents=True, exist_ok=True)
        sample_rate, duration = write_kathbath_audio(row, target)
        prepared.append({'file': str(target), 'language': language_code, 'speaker_id': str(row.get('speaker_id', '')), 'label': 'genuine', 'spoof_generator': '', 'source_dataset': 'ai4bharat/Kathbath', 'split': 'train', 'sample_rate': sample_rate, 'duration': duration})

fields = ['file','language','speaker_id','label','spoof_generator','source_dataset','split','sample_rate','duration']
with metadata_path.open('w', newline='', encoding='utf-8') as handle:
    writer = csv.DictWriter(handle, fieldnames=fields); writer.writeheader(); writer.writerows(prepared)
with provenance_path.open('w', newline='', encoding='utf-8') as handle:
    writer = csv.DictWriter(handle, fieldnames=['dataset_name','source_url','license','language','label','access_conditions','notes']); writer.writeheader()
    writer.writerow({'dataset_name':'ai4bharat/Kathbath','source_url':'https://huggingface.co/datasets/ai4bharat/Kathbath','license':'verify current dataset card before use','language':','.join(LANGUAGE_CONFIGS.values()),'label':'genuine','access_conditions':'gated Hugging Face access','notes':'Streaming pilot; decoded AudioDecoder records; resampled to 16 kHz; no spoof labels'})
print('Prepared genuine records:', len(prepared))

Prepared genuine records: 100


In [5]:
# 7. Generate real TTS spoof samples from Kathbath transcripts.
from pathlib import Path
from transformers import AutoTokenizer, VitsModel

SPOOF_ROWS_PER_LANGUAGE = 20
SPOOF_ROOT = Path(REPO) / 'data' / 'spoof' / 'train' / 'mms-tts'
SPOOF_ROOT.mkdir(parents=True, exist_ok=True)
spoof_metadata = []

for config_name, language_code in LANGUAGE_CONFIGS.items():
    model_id = {'telugu': 'facebook/mms-tts-tel', 'hindi': 'facebook/mms-tts-hin', 'tamil': 'facebook/mms-tts-tam', 'kannada': 'facebook/mms-tts-kan', 'malayalam': 'facebook/mms-tts-mal'}[config_name]
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    print('Loading', model_id, 'on', device)
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    tts_model = VitsModel.from_pretrained(model_id).to(device).eval()
    dataset = load_dataset('ai4bharat/Kathbath', config_name, split='train', streaming=True)
    generated = 0
    for row in dataset:
        if generated >= SPOOF_ROWS_PER_LANGUAGE:
            break
        text = str(row.get('text') or '').strip()
        if not text:
            continue
        target = SPOOF_ROOT / language_code / f'{language_code}_mms_tts_{generated:05d}.wav'
        target.parent.mkdir(parents=True, exist_ok=True)
        if not target.exists():
            inputs = tokenizer(text, return_tensors='pt')
            inputs = {key: value.to(device) for key, value in inputs.items()}
            with torch.inference_mode():
                waveform = tts_model(**inputs).waveform.squeeze().detach().cpu().numpy()
            sf.write(str(target), waveform, int(tts_model.config.sampling_rate), subtype='PCM_16')
        info = sf.info(str(target))
        spoof_metadata.append({'file': str(target), 'language': language_code, 'speaker_id': '', 'label': 'spoof', 'spoof_generator': 'facebook/mms-tts', 'source_dataset': 'ai4bharat/Kathbath transcripts + facebook/mms-tts', 'split': 'train', 'sample_rate': int(info.samplerate), 'duration': float(info.duration)})
        generated += 1
    del tts_model, tokenizer
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

with metadata_path.open('a', newline='', encoding='utf-8') as handle:
    writer = csv.DictWriter(handle, fieldnames=fields)
    writer.writerows(spoof_metadata)
with (DATA_ROOT / 'spoof_provenance.csv').open('w', newline='', encoding='utf-8') as handle:
    writer = csv.DictWriter(handle, fieldnames=['dataset_name','source_url','license','language','label','generator','notes'])
    writer.writeheader()
    writer.writerow({'dataset_name':'Generated from Kathbath transcripts','source_url':'https://huggingface.co/facebook/mms-tts-tel','license':'verify current model card/license before use','language':','.join(LANGUAGE_CONFIGS.values()),'label':'spoof','generator':'facebook/mms-tts','notes':'Pilot synthetic speech; generator family is not suitable as unseen evaluation data'})
print('Generated spoof records:', len(spoof_metadata))

Loading facebook/mms-tts-tel on cuda


Loading weights:   0%|          | 0/762 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/22 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/33 [00:00<?, ?it/s]

Loading facebook/mms-tts-hin on cuda


...

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/762 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/22 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/33 [00:00<?, ?it/s]

Generated spoof records: 100


## 7. Generate a documented Indian-language spoof pilot

This uses one actual TTS generator family, `facebook/mms-tts-*`, for Telugu, Hindi, Tamil, Kannada, and Malayalam. It reuses only Kathbath transcripts, records `spoof_generator=mms-tts`, leaves synthetic `speaker_id` empty, and writes audio at 16 kHz. Increase the row count only after the pilot completes.

In [ ]:
# 8. Start one real Tier 2 pilot job when spoof data is available.
from pathlib import Path
import subprocess

if IN_COLAB:
    from google.colab import drive
    if not Path('/content/drive/MyDrive').exists():
        drive.mount('/content/drive', force_remount=False)
    OUT = Path('/content/drive/MyDrive/veritone-tier2/checkpoints')
else:
    OUT = Path(REPO) / 'checkpoints' / 'tier2'
GenuineDir = Path(REPO) / 'data' / 'kathbath_pilot' / 'genuine'
SpoofDir = Path(REPO) / 'data' / 'spoof' / 'train'
OUT.mkdir(parents=True, exist_ok=True)

real_genuine = list(GenuineDir.rglob('*.wav'))
real_spoof = list(SpoofDir.rglob('*.wav'))
print('genuine WAVs:', len(real_genuine), 'spoof WAVs:', len(real_spoof))

if not real_spoof:
    print('TRAINING NOT STARTED: Kathbath provides genuine speech only.')
    print(f'Upload or mount verified spoof WAVs under {SpoofDir}, then rerun this cell.')
elif not torch.cuda.is_available():
    print('TRAINING NOT STARTED: this runtime has no CUDA GPU; use a Colab T4/L4 runtime.')
else:
    command = [
        sys.executable, '-m', 'voice_detection.train_tier2_ssl',
        '--backbone', 'facebook/wav2vec2-xls-r-300m',
        '--genuine-dir', str(GenuineDir), '--spoof-dir', str(SpoofDir),
        '--epochs', '1', '--batch-size', '1', '--num-workers', '2',
        '--out', str(OUT / 'wav2vec2_xlsr.pt'),
    ]
    result = subprocess.run(command, cwd=REPO)
    if result.returncode != 0:
        raise RuntimeError(f'XLS-R training failed with exit code {result.returncode}')
    checkpoint = OUT / 'wav2vec2_xlsr.pt'
    if not checkpoint.is_file():
        raise FileNotFoundError(f'Training reported success but checkpoint is missing: {checkpoint}')
    print('XLS-R pilot checkpoint:', checkpoint)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
genuine WAVs: 100 spoof WAVs: 100
usage: train_tier2_ssl.py [-h] --backbone BACKBONE --genuine-dir GENUINE_DIR
                          --spoof-dir SPOOF_DIR --out OUT [--epochs EPOCHS]
                          [--batch-size BATCH_SIZE] [--lr LR]
                          [--num-workers NUM_WORKERS] [--val-split VAL_SPLIT]
                          [--crops CROPS] [--synthetic SYNTHETIC]
                          [--seed SEED]
train_tier2_ssl.py: error: unrecognized arguments: --gradient-accumulation 8 --amp
XLS-R pilot checkpoint: /content/drive/MyDrive/veritone-tier2/checkpoints/wav2vec2_xlsr.pt


In [ ]:
# 9. Export XLS-R only after a real checkpoint exists.
import subprocess

checkpoint = OUT / 'wav2vec2_xlsr.pt'
onnx_path = OUT / 'wav2vec2_xlsr.onnx'
if checkpoint.is_file():
    result = subprocess.run([
        sys.executable, '-m', 'voice_detection.export_tier2_ssl',
        '--checkpoint', str(checkpoint), '--out', str(onnx_path),
    ], cwd=REPO)
    if result.returncode != 0:
        raise RuntimeError(f'ONNX export failed with exit code {result.returncode}')
    import onnx
    onnx.checker.check_model(onnx.load(str(onnx_path)))
    print('ONNX export validated:', onnx_path)
else:
    print('Export skipped: no trained XLS-R checkpoint exists.')

Export skipped: no trained XLS-R checkpoint exists.


## 10. Honest completion status

Kathbath genuine data is now loaded and prepared through the verified streaming route. A production anti-spoof ensemble is complete only after:

- real spoof data is supplied and split without speaker/generator leakage;
- XLS-R, WavLM Large, RawNet3, and AASIST are each genuinely trained;
- all four ONNX exports pass numerical equivalence checks;
- calibration uses untouched development data;
- evaluation uses unseen speakers and spoof-generator families.

This notebook never turns Kathbath genuine records into spoof records and never creates placeholder artifacts.